In [0]:
CREATE OR REFRESH STREAMING TABLE techdados.bronze.dlt_customers
COMMENT "Raw orders from cloud storage"
AS
SELECT 
    *,
    current_timestamp() AS ingestion_time,
    _metadata.file_path AS source_file
FROM STREAM cloud_files(
    '/Volumes/techdados/bronze/vol_landing/customer',
    'json',
    map(
        'multiLine', 'true',
        'pathGlobFilter', '*.json'
    )
);

In [0]:
-- Cria tabela customers base
CREATE OR REPLACE MATERIALIZED VIEW techdados.silver.mv_customers
COMMENT 'Silver – customers base'
AS
SELECT
    customer_id,
    name,
    email,
    CAST(created_at AS DATE) AS created_at,
    ingestion_time,
    source_file
FROM techdados.bronze.dlt_customers

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW techdados.silver.mv_customer_addresses
COMMENT 'Silver – exploded customer addresses'
AS
SELECT
    customer_id,
    addr.col.type   AS address_type,
    addr.col.city   AS city,
    addr.col.state  AS state,
    addr.col.zip    AS zip,
    ingestion_time
FROM techdados.bronze.dlt_customers
LATERAL VIEW explode(from_json(addresses, 'array<struct<type:string,city:string,state:string,zip:string>>')) addr

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW techdados.silver.mv_customer_phones
COMMENT 'Silver – exploded customer phones'
AS
SELECT
    customer_id,
    phone.col AS phone_number,
    ingestion_time
FROM techdados.bronze.dlt_customers
LATERAL VIEW explode(from_json(phones, 'array<string>')) phone

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW techdados.silver.mv_customer_preferences
COMMENT 'Silver – exploded customer preferences'
AS
SELECT
    customer_id,
    pref.newsletter AS newsletter,
    pref.sms        AS sms,
    category.col    AS category,
    ingestion_time
FROM (
    SELECT 
        customer_id,
        from_json(preferences, 'struct<newsletter:boolean,sms:boolean,categories:array<string>>') AS pref,
        ingestion_time
    FROM techdados.bronze.dlt_customers
)
LATERAL VIEW explode(pref.categories) category

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW techdados.silver.mv_customer_purchases
COMMENT 'Silver – exploded customer purchase history'
AS
SELECT
    customer_id,
    purchase.col AS purchase_value,
    ingestion_time
FROM techdados.bronze.dlt_customers
LATERAL VIEW explode(from_json(purchase_history, 'array<double>')) purchase